# V-MD3 SAR Data Acquisition

Headless data collection for the V-MD3 radar SAR sweep experiment. This records RADC / RFFT / DONE frames to `.bin` files at each sweep position.

**Before running the notebook:** connect the radar and run the network setup (`run_setup.bat` file as administrator) so the radar responds at `192.168.100.201`. Use the GUI tool (`python measure.py`) separately to view real-time heatmap. This notebook is for acquisition only.

**Workflow:**

- **Cell 1** - Imports and helper functions (run once).
- **Cell 2** - User configuration: session name, streams, frames per position (can edit these).
- **Cell 3** - Connect to the radar (run once per session; re-run only if you change RSET or streams).
- **Cell 4** - Record one Galil position (set the position, then run cell; repeat per position).
- **Cell 5** - Disconnect (run at the end of the session).

The connection stays open between Cell 4 runs, so only reconnect when changing RSET or streams.

## Cell 1 - Imports and helper functions

In [ ]:
import csv
import os
import socket
from datetime import datetime

import numpy as np
import pandas as pd
from lib.vmd3 import VMD3, RdotConfig

# Map user-facing stream names → RdotConfig flags and UDP headers
STREAM_MAP = {
    'radc': RdotConfig.RADC,
    'rfft': RdotConfig.RFFT,
    'done': RdotConfig.DONE,
}
HEADER_MAP = {
    'radc': b'RADC',
    'rfft': b'RFFT',
    'done': b'DONE',
}


def read_any_frame(vmd3, headers):
    """Read the next UDP frame whose header is in `headers`. Returns full frame."""
    while True:
        data = vmd3.recv_udp()
        if data[0:4] not in headers:
            continue
        resp_len = int.from_bytes(data[4:8], byteorder='little')
        while len(data) < resp_len + 8:
            data += vmd3.recv_udp()
        if resp_len != len(data[8:]):
            continue
        return data


def pos_to_str(scan_pos):
    """Convert a position in cm to a filename-safe string (neg5, 0, pos5)."""
    if scan_pos < 0:
        return f'neg{abs(scan_pos)}cm'
    elif scan_pos > 0:
        return f'pos{scan_pos}cm'
    else:
        return '0cm'


print('[cell 1] Imports and helpers loaded.')

## Cell 2 - Configuration

Edit values below for the session, then run the cell. Re-run it any time something is changed.

- **Changing `SESSION_NAME` or `FRAMES_PER_STREAM`** only needs this cell re-run — then go straight to Cell 4. No reconnect needed.
- **Changing `RSET_CONFIG` or `STREAMS`** needs a reconnect: re-run this cell, then Cell 3 (disconnects then reconnects).

`FRAMES_PER_STREAM` is per stream: with 3 streams enabled and a value of 5, you get 5 RADC + 5 RFFT + 5 DONE at each position.

In [ ]:
# ─── Choose where to save binary data (e.g.) ───
OUTPUT_BASE   = "data"             # base directory for all sessions
SESSION_NAME  = "testdatafolder"   # folder name under OUTPUT_BASE, e.g. data/testdatafolder/

# ─── What to record ───
STREAMS = ["radc", "rfft", "done"]   # any of: "radc", "rfft", "done"
FRAMES_PER_STREAM = 5                # frames of EACH selected stream, per position

# ─── Radar mode ───
RSET_CONFIG = 0   # 0 = 2D, 6 m max range, 4.69 cm range bin

# ─── If a position was already recorded ───
ON_EXISTS = "suffix"   # "suffix" = save as _1, _2... | "overwrite" | "skip"

# ─── Connection (no need change) ───
RADAR_IP = "192.168.100.201"
UDP_TIMEOUT = 1.0              # seconds


# ─── Validate & normalize configurations (no need change) ───
STREAMS = [s.strip().lower() for s in STREAMS]
STREAMS = list(dict.fromkeys(STREAMS))           # remove duplicate entries, keep order

_bad = [s for s in STREAMS if s not in STREAM_MAP]
if _bad:
    raise ValueError(f"Unknown stream(s): {_bad}. Valid options: radc, rfft, done")
if not STREAMS:
    raise ValueError("STREAMS is empty — select at least one of: radc, rfft, done")
if FRAMES_PER_STREAM < 1:
    raise ValueError("FRAMES_PER_STREAM must be at least 1")
if not (0 <= RSET_CONFIG <= 8):
    raise ValueError("RSET_CONFIG must be 0-8")
if ON_EXISTS not in ("suffix", "overwrite", "skip"):
    raise ValueError('ON_EXISTS must be "suffix", "overwrite", or "skip"')

SESSION_DIR = os.path.join(OUTPUT_BASE, SESSION_NAME)
BINS_DIR    = os.path.join(SESSION_DIR, "bins")
CSV_PATH    = os.path.join(SESSION_DIR, "scan_log.csv")

# Compute RDOT flag list from the selected streams
SELECTED_RDOT = [STREAM_MAP[s] for s in STREAMS]

print(f"[cell 2] Session folder : {SESSION_DIR}")
print(f"[cell 2] Streams        : {STREAMS}  (RDOT = 0x{sum(r.value for r in SELECTED_RDOT):02X})")
print(f"[cell 2] Frames/stream  : {FRAMES_PER_STREAM} per frame ({FRAMES_PER_STREAM * len(STREAMS)} total per position)")
print(f"[cell 2] RSET mode      : {RSET_CONFIG}")
print(f"[cell 2] On existing    : {ON_EXISTS}")

## Cell 3 - Connect to the radar

Run once at the start of a session. This opens the connection, configures the radar with the RSET mode and selected streams, and creates the session folder.

Safe to re-run: if a connection is already open, it closes it first. Re-run this after changing `RSET_CONFIG` or `STREAMS` in Cell 2.

The connection stays open until you run Cell 5 (or restart the kernel), so you **don't** need to re-run this between Cell 4 recordings.

In [ ]:
# IF a connection is already open from a previous run, close it first so re-running this cell doesn't leave a dangling socket.
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 3] Closed existing connection.")
    except Exception as e:
        print(f"[cell 3] Warning closing old connection: {e}")
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
    vmd3 = None

# ─── Create the session folders + CSV (committed here, not in Cell 2) ───
os.makedirs(BINS_DIR, exist_ok=True)
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filename", "scan_pos_cm", "timestamp_start", "timestamp_stop",
            "duration_s", "n_frames", "rset_mode", "streams", "notes",
        ])
    print(f"[cell 3] Created {CSV_PATH}")
else:
    print(f"[cell 3] Appending to existing {CSV_PATH}")

# ─── Connect ────────────────────────────────────────────────────
print(f"[cell 3] Connecting to V-MD3 at {RADAR_IP}...")
vmd3 = VMD3(tcp_ip=RADAR_IP)
vmd3.connect()
vmd3.set_rset_config(RSET_CONFIG)
vmd3.set_output_config(SELECTED_RDOT)
vmd3.sockUDP.settimeout(UDP_TIMEOUT)

print(f"[cell 3] Ready. RSET={RSET_CONFIG}, streams={STREAMS}, "
      f"RDOT=0x{sum(r.value for r in SELECTED_RDOT):02X}")

## Cell 4 - Record one position

Set `POSITION_CM` to the current position, then run the cell. Repeat for each position in the sweep.

This records `FRAMES_PER_STREAM` frames of **each** selected stream (from Cell 2), saves them to one `.bin` file named by position (`scan_<position>cm.bin`), and logs a row to `scan_log.csv`.

Move the stage, set the position, run this cell. The connection from Cell 3 stays open the whole time.

In [ ]:
POSITION_CM = 5        # ← set to current Galil position, then Shift+Enter
NOTES = ""             # optional note logged to CSV for this position

def record_position(scan_pos, notes=""):
    """Record FRAMES_PER_STREAM of each selected stream at one position."""
    accepted = tuple(HEADER_MAP[s] for s in STREAMS)
    target_per_stream = FRAMES_PER_STREAM

    # ─── Build the output path, depends on ON_EXISTS ───
    base_name = f"scan_{pos_to_str(scan_pos)}"
    filepath = os.path.join(BINS_DIR, base_name + ".bin")

    if os.path.exists(filepath):
        if ON_EXISTS == "skip":
            print(f"[cell 4] {base_name}.bin already exists — SKIPPED "
                  f"(ON_EXISTS='skip').")
            return
        elif ON_EXISTS == "overwrite":
            print(f"[cell 4] {base_name}.bin exists — overwriting.")
        elif ON_EXISTS == "suffix":
            i = 1
            while os.path.exists(os.path.join(BINS_DIR, f"{base_name}_{i}.bin")):
                i += 1
            filepath = os.path.join(BINS_DIR, f"{base_name}_{i}.bin")
            print(f"[cell 4] {base_name}.bin exists — saving as "
                  f"{os.path.basename(filepath)}.")

    filename = os.path.basename(filepath)

    # ─── Record ───
    per_stream = {s: 0 for s in STREAMS}   # how many of each we have
    header_to_name = {HEADER_MAP[s]: s for s in STREAMS}
    n_written = 0
    bytes_written = 0

    # ─── Align to a cycle boundary so recording starts on RADC ───
    if "radc" in STREAMS:
        # Step 1: drain all buffered frames (short timeout = "nothing left")
        _old_timeout = vmd3.sockUDP.gettimeout()
        vmd3.sockUDP.settimeout(0.05)
        try:
            while True:
                try:
                    vmd3.recv_udp()
                except socket.timeout:
                    break
        finally:
            vmd3.sockUDP.settimeout(_old_timeout)
        # Step 2: grab the next fresh RADC and KEEP it as frame 1
        first_frame = None
        while True:
            try:
                first_frame = read_any_frame(vmd3, (b'RADC',))
                break
            except socket.timeout:
                continue
    else:
        first_frame = None

    start_time = datetime.now()

    print(f"[cell 4] Recording at {scan_pos:+d} cm → {filename}  "
          f"(need {target_per_stream} of each: {STREAMS})")

    out = open(filepath, "wb")
    try:
        # Write the aligning RADC as frame 1
        if first_frame is not None:
            out.write(first_frame)
            per_stream["radc"] += 1
            n_written += 1
            bytes_written += len(first_frame)
            counts_str = "  ".join(
                f"{s}:{per_stream[s]}/{target_per_stream}" for s in STREAMS)
            print(f"   RADC   {counts_str}")
            
        # Keep reading until every selected stream has hit its target count
        while any(per_stream[s] < target_per_stream for s in STREAMS):
            try:
                frame = read_any_frame(vmd3, accepted)
            except socket.timeout:
                continue

            name = header_to_name[frame[0:4]]

            # Only save frames we still need (don't over-collect one stream)
            if per_stream[name] < target_per_stream:
                out.write(frame)
                per_stream[name] += 1
                n_written += 1
                bytes_written += len(frame)
                counts_str = "  ".join(
                    f"{s}:{per_stream[s]}/{target_per_stream}" for s in STREAMS)
                print(f"   {name.upper()}   {counts_str}")
    finally:
        out.close()

    stop_time = datetime.now()
    duration = (stop_time - start_time).total_seconds()

    # ─── Log to CSV ───
    with open(CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            filename, scan_pos,
            start_time.isoformat(timespec="seconds"),
            stop_time.isoformat(timespec="seconds"),
            f"{duration:.2f}", n_written, RSET_CONFIG,
            "+".join(STREAMS), notes or "",
        ])

    print(f"[cell 4] Saved {filename}: {n_written} frames "
          f"({bytes_written/1e6:.2f} MB, {duration:.1f} s)")

# ─── Run it for the position set above ───
record_position(POSITION_CM, NOTES)

## Cell 5 - Disconnect

Run at the end of a session to close the radar connection cleanly.

Safe to run even if nothing is connected since it just reports that there's nothing to close.

In [ ]:
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 5] Disconnected from V-MD3.")
    except Exception as e:
        print(f"[cell 5] Disconnect warning: {e}")
        # Force-close the sockets so the OS releases the ports
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
        print("[cell 5] Sockets force-closed.")
    finally:
        vmd3 = None
else:
    print("[cell 5] Nothing to disconnect (no open connection).")

## Cell 6 - Convert .bin to long-format CSV

Decodes a saved `.bin` file into a lossless "long" CSV: one row per (frame, bin, chirp, channel) value, with complex I/Q split into separate `I` and `Q` columns.

Set `BIN_TO_CONVERT` to the file you want, choose which streams to export, then run.

Set `OUTPUT_CSV_PATH` to the directory and file to save csv data.

Warning: long format is large - one RADC or RFFT frame is 32,768 rows, so a 5-frame-each file is ~330,000 rows. Lossless but big.

In [ ]:
# ─── What to convert ───
BIN_TO_CONVERT = os.path.join(BINS_DIR, "scan_0cm.bin")   # path to a .bin file
CONVERT_STREAMS = ["radc", "rfft"]     # which streams to export: radc and/or rfft
CONVERT_MODE = "2D"                    # "2D" or "3D" (matches how it was recorded)

# ─── Where to save ───
OUTPUT_CSV_PATH = "C:/My Computer/vmd3/test/data/csv/scan_0cm.csv"


# ─── Decoders ───
def _get_frames(filepath, header, expected_len):
    """Extract all valid payloads for a given header from a .bin file."""
    with open(filepath, "rb") as f:
        raw = f.read()
    frames, start = [], 0
    while True:
        idx = raw.find(header, start)
        if idx == -1:
            break
        if idx + 8 > len(raw):
            break
        payload_len = int.from_bytes(raw[idx+4:idx+8], byteorder="little")
        if payload_len == expected_len:
            p0, p1 = idx + 8, idx + 8 + expected_len
            if p1 <= len(raw):
                payload = raw[p0:p1]
                if header not in payload:   # reject header-in-payload false hits
                    frames.append(payload)
        start = idx + 1
    return frames


def _decode_radc_2d(frame):
    raw = np.frombuffer(frame, dtype="<i2").reshape(64, 4, 256)
    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)
    complex_data = i + 1j * q                       # (chirp=64, ch=4, samples=128)
    return np.transpose(complex_data, (2, 0, 1))    # (samples, chirps, channels)


def _decode_rfft_2d(frame):
    raw = np.frombuffer(frame, dtype="<i2").reshape(4, 128, 64, 2)
    i = raw[:, :, :, 1].astype(np.float64)
    q = raw[:, :, :, 0].astype(np.float64)
    complex_data = i + 1j * q                      # (ch, range_bin, chirp)
    return np.transpose(complex_data, (1, 2, 0))   # (range_bin, chirp, ch)


# ─── Build the long dataframe ───
def bin_to_long_df(filepath, streams, mode):
    expected_len = 131072 if mode == "2D" else 196608
    header_map = {"radc": b"RADC", "rfft": b"RFFT"}
    all_rows = []

    for s in streams:
        header = header_map[s]
        frames = _get_frames(filepath, header, expected_len)
        if not frames:
            print(f"  [{s}] no frames found — skipping")
            continue

        for f_idx, frame in enumerate(frames):
            if s == "radc":
                cube = _decode_radc_2d(frame)
            else:   # rfft 
                cube = _decode_rfft_2d(frame)

            # cube shape: (bin, chirp, channel) — flatten to index arrays
            n_bin, n_chirp, n_chan = cube.shape
            bb, cc, hh = np.meshgrid(
                np.arange(n_bin), np.arange(n_chirp), np.arange(n_chan),
                indexing="ij"
            )
            df = pd.DataFrame({
                "file":    os.path.basename(filepath),
                "stream":  s,
                "frame":   f_idx,
                "bin":     bb.ravel(),   # sample index (radc) or range bin (rfft)
                "chirp":   cc.ravel(),
                "channel": hh.ravel(),
                "I":       np.real(cube).ravel(),
                "Q":       np.imag(cube).ravel(),
            })
            all_rows.append(df)
            print(f"  [{s}] frame {f_idx}: {len(df)} rows")

    if not all_rows:
        print("No data converted.")
        return None
    return pd.concat(all_rows, ignore_index=True)


# ─── Run ────────────────────────────────────────────────────────
print(f"[cell 6] Converting {os.path.basename(BIN_TO_CONVERT)} "
      f"(streams={CONVERT_STREAMS}, mode={CONVERT_MODE})...")

df = bin_to_long_df(BIN_TO_CONVERT, CONVERT_STREAMS, CONVERT_MODE)

if df is not None:
    # Decide output path: manual override, or auto next to the .bin
    if OUTPUT_CSV_PATH.strip():
        out_csv = OUTPUT_CSV_PATH.strip()
        out_dir = os.path.dirname(out_csv)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
    else:
        out_csv = os.path.splitext(BIN_TO_CONVERT)[0] + "_long.csv"

    df.to_csv(out_csv, index=False)
    print(f"[cell 6] Wrote {out_csv}")
    print(f"[cell 6] Shape: {df.shape[0]} rows × {df.shape[1]} cols")

    # ─── Per-stream row breakdown ───────────────────────────────
    stream_counts = df["stream"].value_counts()
    for s in CONVERT_STREAMS:
        n = int(stream_counts.get(s, 0))
        print(f"[cell 6]   {s}: {n} rows")
    print(f"[cell 6]   total: {len(df)} rows")

    print(df.head())